# RAG: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking

**Document:** *IIIT-Delhi Regulations for B.Tech. Programs* (Version 13, May 2024). This is the official
21-page academic rulebook of an Indian institute, published at
<https://iiitd.ac.in/sites/default/files/docs/education/2024/2024-May-UG%20Regulations.pdf>.

**Why this document and not a popular rulebook (e.g. the FIDE Laws of Chess)?** Popular rulebooks are all over
the LLMs' pre-training data, so the model can answer correctly even when retrieval fails, and the difference between
Answer 1 and Answer 2 gets hidden. An institute's own regulations are niche: the numbers (156 credits, F = 2 grade
points, CGPA > 8.0 for overload, four semester leaves …) are specific to IIIT-Delhi and differ from other colleges.
Section 5 checks this directly by asking the LLMs the same questions **without any context** (closed-book).
The document is full of exceptions ("an exception may be made for the graduating batch…"), prohibitions
("first year students cannot take any overload") and near-duplicate vocabulary (*credits*, *CGPA*, *semester*
appear in almost every section), which is what makes dense retrieval stumble.

*(IIIT-Hyderabad's own regulations live on intranet.iiit.ac.in, which is only reachable from the campus network.
The pipeline is document-agnostic: point `PDF_PATH` at another regulations PDF and re-label the queries.)*

**What this notebook does**

| Step | Pipeline 1: Baseline | Pipeline 2: Re-Ranking |
|---|---|---|
| Retrieve | Bi-encoder (`all-MiniLM-L6-v2`) + FAISS inner product → Top 10 | *same Top 10 candidates* |
| Select | Top 3 by cosine similarity | Cross-encoder (`ms-marco-MiniLM-L-6-v2`) scores all 10 (query, chunk) pairs → re-sort → Top 3 |
| Generate | Local Qwen LLMs → **Answer 1** | Same LLMs, identical prompt → **Answer 2** |

Two local **Qwen** models are used as generators, so the effect of retrieval can be seen at two model scales:

* `Qwen2.5-1.5B-Instruct`: small, run in-process with Hugging Face `transformers` on CPU
* `Qwen3.8-27B` (GGUF, Q3_K_XL): large, served on the GPU by `llama.cpp`'s `llama-server` (OpenAI-compatible API)

Sections:
1. Setup & configuration
2. Load and chunk the PDF
3. Pipeline 1: bi-encoder index + baseline retrieval
4. Pipeline 2: cross-encoder re-ranking
5. LLMs + shared prompt + closed-book check (does the LLM already know this document?)
6. Failure-mode analysis (stress tests): rank shifts + Answer 1 vs Answer 2
7. Aggregate retrieval evaluation over 151 labelled questions
8. Latency profiling
9. Scaling experiment (why not cross-encode everything?)
10. Save results

## 1. Setup & configuration

In [1]:
# Install dependencies (uncomment on a fresh environment)
# %pip install pymupdf sentence-transformers faiss-cpu transformers accelerate torch pandas requests

import os, re, time, json, random, warnings
import numpy as np
import pandas as pd
import pymupdf                   # PDF text extraction
import faiss                     # vector store (exact inner-product index)
import requests                  # talks to the local llama.cpp server
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# ---- Configuration -------------------------------------------------------
PDF_PATH        = "data/IIITD_BTech_Regulations_2024.pdf"
BI_ENCODER      = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder
CROSS_ENCODER   = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker
CHUNK_MAX_CHARS = 600     # target chunk size (characters)
TOP_K_RETRIEVE  = 10      # candidates from the bi-encoder
TOP_K_CONTEXT   = 3       # chunks passed to the LLM
MAX_NEW_TOKENS  = 220
SEED            = 42

# Local Qwen generators. The small one runs in-process; the large one is a llama.cpp server started with e.g.
#   llama-server -m Qwen3.8-27B-UD-Q3_K_XL.gguf --alias qwen3.8-27b --port 8081 -ngl 99 -c 32768 --jinja
SMALL_LLM       = "Qwen/Qwen2.5-1.5B-Instruct"
LLAMA_SERVER    = os.environ.get("LLAMA_SERVER", "http://127.0.0.1:8081/v1")
LARGE_LLM       = "qwen3.8-27b"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
print("torch", torch.__version__, "| encoders + small LLM on:", DEVICE)

/home/akshat/Shared/sem 7/LMA/assignment/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.14.1+cpu | encoders + small LLM on: cpu


## 2. Load and chunk the PDF

The regulations are organised as numbered sections (`5.8 Academic Load`, `7.4 Attendance and Leave of Absence` …)
containing numbered paragraphs and list items (`(1)`, `a)`, `•`, `1.`). Instead of fixed windows I chunk on that structure:

1. Extract text with PyMuPDF and drop the page-number line of every page (the grade table also contains bare
   numbers such as `F (Fail) 2`, so only the page number itself is removed).
2. Track the current section. Top-level sections must appear in order (1, 2, 3 …), which stops table cells and
   page numbers from being read as headings. Sub-section headings may have the number and title on separate lines.
3. Start a new *paragraph* at every list marker or sub-heading (`(2)`, `b)`, `•`, `Eligibility to …`, `Version 10 …`).
4. Greedily pack consecutive paragraphs of the **same section** into chunks of ≤ ~600 characters
   (a chunk never crosses a section boundary); over-long paragraphs are split on sentence boundaries.
5. Each chunk is prefixed with `[Section 5.8: Academic Load]` so the encoders and the LLM know where it comes from.

The *Change History* at the end is kept on purpose. It mentions old rules (e.g. "minimum credit requirement for
graduation increased to **152**", while the current rule is **156**) and acts as a natural distractor.

In [2]:
# Paragraph / list-item starts inside a section (numbered items, bullets, and the document's un-numbered sub-headings)
PARA_RE = re.compile(
    r"^(\(\d+\)|\([ivx]+\)|[a-e]\)|\d\.\s|•|-\s*$|-\s|Eligibility to|Tie Breaking|Method:|General Rules|Credits:"
    r"|Grades:|Number of Seats|Admission Criteria|Reservations$|Offer of Admission|Cancellation of Admission"
    r"|Foreign Students|Non-degree Students|Version \d+|Letter Grade|June 21|September 21|January 21|Students under"
    r"|Students can avail|Students willing|Only eligible|When a student|When the student|Students eligible"
    r"|Repeat of a course|Results of students|Under exceptional|A student who has completed)")


def load_pdf_lines(path):
    # Extract text page by page and drop the page-number line printed on each page.
    doc = pymupdf.open(path)
    lines = []
    for pno, page in enumerate(doc):
        page_lines = [l.strip() for l in page.get_text().split("\n")]
        if str(pno + 1) in page_lines:
            page_lines.remove(str(pno + 1))          # first occurrence on the page = its page number
        lines += page_lines
    print(f"Pages: {doc.page_count} | lines: {len(lines):,}")
    return lines


def split_into_paragraphs(lines):
    # Walk the lines, tracking the current section and starting a new paragraph at every list marker.
    lines = [re.sub(r"\s+", " ", l).strip() for l in lines]
    lines = [l for l in lines if l and not l.startswith(("REGULATIONS FOR B.TECH", "For ORDINANCES", "https://"))]
    sec_id, sec_title, next_top = "0", "Preamble", 1
    paras, cur, i = [], None, 0

    def flush():
        nonlocal cur
        if cur and cur["text"].strip():
            paras.append(cur)
        cur = None

    while i < len(lines):
        s = lines[i]
        nxt = lines[i + 1] if i + 1 < len(lines) else ""
        # Top-level section, e.g. "5 Registration" (number and title may be on separate lines).
        # Requiring the numbers to come in order filters out page numbers and grade-table cells.
        m = re.fullmatch(r"(\d{1,2})(?: ([A-Z][A-Za-z ,]+))?", s)
        if m and int(m.group(1)) == next_top and (m.group(2) or re.fullmatch(r"[A-Z][A-Za-z ,]+", nxt)):
            flush(); sec_id, sec_title = m.group(1), (m.group(2) or nxt); next_top += 1
            i += 1 if m.group(2) else 2
            continue
        # Sub-section "6.4 Change of Grade ...", or "6.4" alone with the title on the next line
        m = re.match(r"^(\d{1,2}\.\d{1,2})(?: (.*))?$", s)
        if m and m.group(1).split(".")[0] == sec_id.split(".")[0]:
            flush()
            title = m.group(2) or nxt
            if not m.group(2): i += 1
            if nxt and nxt[0].islower() and m.group(2):          # heading wrapped onto a second line
                title += " " + nxt; i += 1
            sec_id = m.group(1)
            if title.startswith("("):                             # "6.5" + "(i) Grade Improvement"
                sec_title = re.sub(r"^\(\w+\)\s*", "", title)
                cur = {"sec": sec_id, "title": sec_title, "text": title}
            elif len(title) > 60:                                 # 6.8 / 6.9: the "heading" is the clause itself
                sec_title = title[:60].rsplit(" ", 1)[0] + " ..."
                cur = {"sec": sec_id, "title": sec_title, "text": title}
            else:
                sec_title = title
            i += 1
            continue
        if s.startswith("Change History"):
            flush(); sec_id, sec_title = "CH", "Change History"; i += 1
            continue
        if cur is None or PARA_RE.match(s):                       # a new paragraph / list item starts here
            flush(); cur = {"sec": sec_id, "title": sec_title, "text": s}
        else:                                                     # continuation line
            cur["text"] += " " + s
        i += 1
    flush()
    return paras


def pack_chunks(paras, max_chars=CHUNK_MAX_CHARS):
    # Greedily merge consecutive paragraphs of the same section up to max_chars.
    chunks, buf = [], None
    for p in paras:
        if buf and buf["sec"] == p["sec"] and len(buf["text"]) + len(p["text"]) <= max_chars:
            buf["text"] += "\n" + p["text"]
        else:
            if buf: chunks.append(buf)
            buf = dict(p)
    chunks.append(buf)

    out = []
    for c in chunks:                                              # split over-long paragraphs on sentence ends
        if len(c["text"]) <= max_chars * 1.4:
            out.append(c); continue
        piece = ""
        for sent in re.split(r"(?<=[.;:])\s+", c["text"]):
            if piece and len(piece) + len(sent) > max_chars:
                out.append({**c, "text": piece.strip()}); piece = ""
            piece += " " + sent
        if piece.strip(): out.append({**c, "text": piece.strip()})

    for i, c in enumerate(out):
        c["id"] = i
        c["passage"] = f"[Section {c['sec']}: {c['title']}]\n{c['text']}"   # what is embedded / shown to the LLM
    return out


lines    = load_pdf_lines(PDF_PATH)
paras    = split_into_paragraphs(lines)
chunks   = pack_chunks(paras)
passages = [c["passage"] for c in chunks]

lens = [len(p) for p in passages]
print(f"Paragraphs: {len(paras)} | Chunks: {len(chunks)} | chars/chunk min={min(lens)} "
      f"median={int(np.median(lens))} max={max(lens)}")
pd.DataFrame([{"id": c["id"], "section": c["sec"], "title": c["title"][:40], "chars": len(c["passage"]),
               "starts with": c["text"][:70]} for c in chunks]).iloc[[0, 29, 31, 42, 69, 80, 90]]

Pages: 21 | lines: 944
Paragraphs: 219 | Chunks: 98 | chars/chunk min=160 median=504 max=855


,id,section,title,chars,starts with
0,0,1,General,257,(1) This document gives the general regulations applicable to all B.Te
29,29,5.7,Summer term registration,666,Registering in the summer term is optional. A student may register for
31,31,5.8,Academic Load,585,When a student registers for more credits than the stipulated normal l
42,42,6.3,Grading Scheme,504,Letter Grade Grade Points Remarks A+ (Outstanding) 10 Letter grades A+
69,69,7.4,Attendance and Leave of Absence,639,"(2) A student may be allowed to take semester leave, with permission b"
80,80,8.1,Minimum and Maximum Duration and Credits,514,The minimum duration for completing the B.Tech. program is 8 semesters
90,90,CH,Change History,580,"Version 2 of the UG Manual, applicable from 2010. Main changes: Organi"


In [3]:
# Peek at one chunk to see exactly what the encoders and the LLM receive
print(chunks[31]["passage"])

[Section 5.8: Academic Load]
When a student registers for more credits than the stipulated normal load, it is called the overload. First year students cannot take any overload. Second year students having a CGPA of more than 8.0 can take an overload of up to 2 additional credits over 20 credits. Third- and fourth-year students having a CGPA of more than 8.0 can take an overload of up to 2 additional credits over 22 credits. If a student is not having CGPA more than 8.0 may be allowed to take an overload only in one semester in the final year, i.e., in either 7th or 8th semester.


## 3. Pipeline 1: Baseline retrieval (bi-encoder)

The bi-encoder embeds every chunk **independently** into a 384-d vector. Embeddings are L2-normalised, so the
inner product in `faiss.IndexFlatIP` equals cosine similarity. At query time only the query is embedded and
compared against the stored vectors.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(passages, batch_size=32, normalize_embeddings=True,
                              convert_to_numpy=True).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])     # exact inner-product search
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]} in {time.perf_counter()-t0:.2f}s (one-off cost)")


def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Pipeline 1, steps a-b: embed the query and return the top-k chunk ids with cosine scores.
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return ids[0].tolist(), scores[0].tolist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25414.04it/s]

Indexed 98 chunks, dim=384 in 0.58s (one-off cost)


## 4. Pipeline 2: Cross-encoder re-ranking

The cross-encoder reads **query and chunk together** as one sequence (`[CLS] query [SEP] chunk [SEP]`), so every
query token can attend to every chunk token. It outputs one relevance logit per pair. It only re-scores the
10 bi-encoder candidates.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device=DEVICE)


def rerank(query, cand_ids):
    # Pipeline 2, steps b-c: score each (query, chunk) pair jointly and sort descending.
    pairs = [(query, passages[i]) for i in cand_ids]
    ce_scores = cross_encoder.predict(pairs, batch_size=16)
    order = np.argsort(-ce_scores)
    return [cand_ids[j] for j in order], [float(ce_scores[j]) for j in order]


def run_both_retrievals(query):
    # Pipeline 1 top-3 and Pipeline 2 top-3 from the SAME top-10 candidate set.
    cand_ids, bi_scores = baseline_retrieve(query)
    rr_ids, ce_scores = rerank(query, cand_ids)
    return {"cand_ids": cand_ids, "bi_scores": bi_scores,
            "rr_ids": rr_ids, "ce_scores": ce_scores,
            "baseline_top3": cand_ids[:TOP_K_CONTEXT], "rerank_top3": rr_ids[:TOP_K_CONTEXT]}

# quick smoke test
r = run_both_retrievals("Can a first-year student take an overload?")
print("baseline top-3 sections:", [chunks[i]["sec"] for i in r["baseline_top3"]])
print("re-ranked top-3 sections:", [chunks[i]["sec"] for i in r["rerank_top3"]])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 18694.37it/s]

baseline top-3 sections: ['5.8', '5.8', '5.8']
re-ranked top-3 sections: ['5.8', '5.8', '5.8']


## 5. LLMs, the shared prompt, and a closed-book check

Both pipelines use **the same LLM, the same prompt template and greedy decoding** (temperature 0), so any difference
between Answer 1 and Answer 2 comes only from *which 3 chunks* were retrieved. Every answer is generated by both Qwen
models:

* **Qwen2.5-1.5B-Instruct**: loaded with `transformers`, runs on CPU.
* **Qwen3.8-27B**: served by `llama-server` on the GPU and called through its OpenAI-compatible `/v1/chat/completions`
  endpoint (thinking mode disabled). If the server is not running, this model is skipped and the notebook still runs.

In [6]:
SYSTEM_PROMPT = (
    "You are an assistant that answers questions about the IIIT-Delhi Regulations for B.Tech. Programs. "
    "Answer ONLY using the context passages provided. Cite the section numbers you rely on. "
    "If the context does not contain the answer, say: 'The provided context does not answer this.' "
    "Keep the answer to 2-4 sentences.")


def build_prompt(query, chunk_ids):
    # Identical template for both pipelines; only the context changes.
    context = "\n\n".join(f"[Passage {n}]\n{passages[i]}" for n, i in enumerate(chunk_ids, 1))
    user = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]


class HFChat:
    # Small Qwen model run in-process with transformers (greedy decoding).
    def __init__(self, name):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForCausalLM.from_pretrained(
            name, dtype=torch.bfloat16 if DEVICE == "cpu" else torch.float16).to(DEVICE).eval()

    @torch.no_grad()
    def __call__(self, msgs):
        enc = self.tok.apply_chat_template(msgs, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(DEVICE)
        t0 = time.perf_counter()
        out = self.model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        dt = time.perf_counter() - t0
        new = out[0, enc["input_ids"].shape[1]:]
        return self.tok.decode(new, skip_special_tokens=True).strip(), dt, len(new)


class LlamaServerChat:
    # Large Qwen model behind llama.cpp's OpenAI-compatible server (temperature 0 = greedy).
    def __init__(self, base_url, model):
        self.url, self.model = base_url.rstrip("/") + "/chat/completions", model
        requests.get(base_url.rstrip("/") + "/models", timeout=5).raise_for_status()   # fail fast if not running

    def __call__(self, msgs):
        t0 = time.perf_counter()
        r = requests.post(self.url, timeout=600, json={
            "model": self.model, "messages": msgs, "temperature": 0, "seed": SEED,
            "max_tokens": MAX_NEW_TOKENS, "chat_template_kwargs": {"enable_thinking": False}}).json()
        dt = time.perf_counter() - t0
        return r["choices"][0]["message"]["content"].strip(), dt, r["usage"]["completion_tokens"]


LLMS = {"Qwen2.5-1.5B": HFChat(SMALL_LLM)}
try:
    LLMS["Qwen3.8-27B"] = LlamaServerChat(LLAMA_SERVER, LARGE_LLM)
except Exception as e:
    print(f"llama-server not reachable at {LLAMA_SERVER} ({type(e).__name__}); skipping the 27B model")
print("LLMs:", list(LLMS))


def generate_answer(llm_name, query, chunk_ids):
    # Returns (answer text, generation seconds, generated tokens) for one model and one context.
    return LLMS[llm_name](build_prompt(query, chunk_ids))


q = "How many grade points does an F grade carry?"
print(generate_answer("Qwen2.5-1.5B", q, run_both_retrievals(q)["rerank_top3"])[0])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 22727.52it/s]

LLMs: ['Qwen2.5-1.5B', 'Qwen3.8-27B']


An F grade carries 2 grade points.


### Closed-book check: does the LLM already know this document?

If the LLM could answer from memory, retrieval quality would not matter much and Answer 1 vs Answer 2 would look
the same. Here both models get the questions **with no context** (a plain "answer from your own knowledge" prompt).
The ground truth is in the document: 156 credits, an F carries 2 points, overload needs CGPA > 8.0, at most four
semester leaves, and so on.

In [7]:
CLOSED_BOOK_SYSTEM = ("Answer the question about IIIT-Delhi's B.Tech. regulations from your own knowledge "
                      "in 1-2 sentences. Give specific numbers if you know them.")
CLOSED_BOOK_QS = [
    ("How many credits are required to graduate with a B.Tech. degree at IIIT-Delhi?", "156 credits (Sec 8.1)"),
    ("How many grade points does an F grade carry at IIIT-Delhi?",                     "2 points (Sec 6.3)"),
    ("What CGPA does an IIIT-Delhi second-year student need to take an overload?",    "more than 8.0 (Sec 5.8)"),
    ("How many semester leaves can an IIIT-Delhi B.Tech. student take?",              "at most four (Sec 7.4)"),
    ("What SGPA puts an IIIT-Delhi student on academic warning?",                      "4.5 or less (Sec 7.2)"),
]
closed_rows = []
for q, truth in CLOSED_BOOK_QS:
    row = {"question": q, "document says": truth}
    for name, llm in LLMS.items():
        row[name] = llm([{"role": "system", "content": CLOSED_BOOK_SYSTEM}, {"role": "user", "content": q}])[0]
    closed_rows.append(row)
closed_df = pd.DataFrame(closed_rows)
with pd.option_context("display.max_colwidth", 200):
    display(closed_df)

,question,document says,Qwen2.5-1.5B,Qwen3.8-27B
0,How many credits are required to graduate with a B.Tech. degree at IIIT-Delhi?,156 credits (Sec 8.1),IIIT-Delhi requires approximately 300 credits for graduation with a Bachelor of Technology (B.Tech) degree.,IIIT-Delhi requires a total of 160 credits to graduate with a B.Tech. degree.
1,How many grade points does an F grade carry at IIIT-Delhi?,2 points (Sec 6.3),An F grade carries 0.5 grade points at IIIT-Delhi.,An F grade carries 0 grade points at IIIT-Delhi.
2,What CGPA does an IIIT-Delhi second-year student need to take an overload?,more than 8.0 (Sec 5.8),An IIIT-Delhi second-year student needs a minimum of 7.5 CGPA to be eligible for taking an overload.,A second-year student at IIIT-Delhi typically needs a CGPA of at least 8.00 to be eligible for taking an overload.
3,How many semester leaves can an IIIT-Delhi B.Tech. student take?,at most four (Sec 7.4),An IIIT-Delhi B.Tech. student is allowed up to two semester breaks per academic year.,IIIT-Delhi B.Tech. students are generally allowed to take a maximum of **two** semester leaves (one for each academic year) during their four-year program.
4,What SGPA puts an IIIT-Delhi student on academic warning?,4.5 or less (Sec 7.2),An IIIT Delhi student gets put on academic warning with a SGPA of 3.0.,An IIIT-Delhi student is placed on academic warning if their SGPA is less than 6.0.


## 6. Failure-mode analysis (stress tests)

Each test query has **gold phrase(s)**: the verbatim text of the clause that answers it, checked by reading the
regulations. A chunk is *gold* if it contains one of them. For every query I report:

* the gold chunk's rank under the bi-encoder (Pipeline 1) and after cross-encoder re-ranking (Pipeline 2),
* the full top-10 table with both scores,
* **Answer 1** (baseline top-3) vs **Answer 2** (re-ranked top-3) from **both** Qwen models,
* a **correctness verdict** for each answer: I wrote a reference answer for every query from the regulation text
  (`expected` below), and the large Qwen model grades each answer against it (CORRECT / INCORRECT). A plain
  string match did not work for yes/no questions (the model echoes the question's words), so a reference-based
  judge is used instead. I checked the verdicts by reading the answers; they are shown next to each answer.

The queries came out of a sweep of the labelled set in section 7. I kept the ones where the bi-encoder buries the gold
clause at ranks 4–10 and the cross-encoder lifts it into the top 3, plus two **counter-examples** where re-ranking
makes things worse and one **limitation** where neither pipeline gets the gold chunk into the top 3.

In [8]:
FAILURE_QUERIES = [
    {"category": "Keyword trap / distractor",
     "query": "My CGPA is 7.5. Can I take an extra course in my 5th semester?",
     "gold": ["overload only in one semester in the final year"],
     "expected": "No. An overload needs a CGPA of more than 8.0; a student without that may overload only in one semester of the final year (7th or 8th), so not in the 5th semester.",
     "why": "'Extra course' and 'CGPA' match Sec 6.8 / 6.9 (extra credits beyond 156, extra courses to improve CGPA) "
            "and the Change History. The real rule is the OVERLOAD clause in Sec 5.8 (CGPA > 8.0, otherwise only "
            "in the final year), which never says 'extra course'."},
    {"category": "Keyword trap / distractor",
     "query": "After finishing 156 credits, for how many extra semesters can I stay to improve my CGPA?",
     "gold": ["will be allowed in only one extra semester"],
     "expected": "Only one extra semester (registering for at most two extra courses).",
     "why": "'156 credits' and 'CGPA' are all over Sec 6.8 (the extra-credit CGPA method, four chunks). "
            "The answer ('only one extra semester') is in Sec 6.9."},
    {"category": "Keyword trap / synonyms",
     "query": "Can I take a gap semester to work on my startup?",
     "gold": ["entrepreneurship"],
     "expected": "Yes, as a semester leave (entrepreneurship is an accepted reason), with permission before the last date of add/drop; at most four semester leaves in the program.",
     "why": "The document says 'semester leave' and 'entrepreneurship', never 'gap semester' or 'startup'."},
    {"category": "Keyword trap / synonyms",
     "query": "Can I spend a year studying at a foreign university?",
     "gold": ["spend up to 2 semesters at another University"],
     "expected": "Yes, a student may spend up to 2 semesters at another university (credit transfer capped at 40 credits, 20 per semester).",
     "why": "'Foreign' pulls in the admissions paragraph on 'Foreign Students' (Sec 3). The rule is Sec 6.7: "
            "'places of repute outside IIIT-Delhi … up to 2 semesters at another University'."},
    {"category": "Exception / corner case",
     "query": "Does the six-year maximum duration include semester leaves?",
     "gold": ["including the semester in which student took semester leave"],
     "expected": "Yes. The six-year (12-semester) maximum includes semesters taken as leave (unlike the 8-semester minimum, which excludes them).",
     "why": "Sec 8.1 has the twist: the 8-semester MINIMUM excludes leave semesters, but the six-year MAXIMUM "
            "includes them. The bi-encoder returns generic leave chunks (Sec 7.4) and the semester calendar."},
    {"category": "Exception / corner case",
     "query": "Does an online course count as an in-class course during my internship semester?",
     "gold": ["register for 4 credits of only IP/IS/UR/BTP/OC courses"],
     "expected": "No. During the internship semester the student registers for 4 credits of only IP/IS/UR/BTP/OC courses and may not take any in-class course, so an online course is allowed and is not treated as an in-class course.",
     "why": "General online-course rules (Sec 4) dominate. The internship exception (Sec 5.8) says interns may "
            "register ONLY for 4 credits of IP/IS/UR/BTP/OC and no in-class course."},
    {"category": "Negation & constraints",
     "query": "How many semester leaves are allowed in the whole program?",
     "gold": ["more than four semester leave"],
     "expected": "At most four semester leaves in the entire program; more may lead to automatic termination.",
     "why": "The limit is phrased as a prohibition: 'shall not be permitted more than four semester leaves' / "
            "'more than four … may result in automatic termination'."},
    {"category": "Negation & constraints",
     "query": "Is a student expelled from another institute on disciplinary grounds eligible for admission?",
     "gold": ["on disciplinary grounds are not eligible"],
     "expected": "No, a candidate whose program was terminated elsewhere on disciplinary grounds is not eligible for admission.",
     "why": "'Disciplinary' pulls Sec 9 (Disciplinary Action) to rank 1. The answer is a negated eligibility rule "
            "inside Sec 3 Admission."},
    {"category": "Other: vital chunk buried at rank 4-10",
     "query": "When is a student's program terminated for poor academic performance?",
     "gold": ["low SGPA (of <4.0) in two consecutive semesters"],
     "expected": "When the student gets an academic warning for low SGPA (below 4.0) in two consecutive semesters and/or gets two or more F grades in each of two consecutive semesters.",
     "why": "Sec 7 (general 'may be terminated'), leave-termination and appeals chunks outrank the actual rule "
            "in Sec 7.3."},
    # ---- counter-examples: re-ranking hurts ---------------------------------------------------
    {"category": "Counter-example: re-ranking hurts",
     "query": "Does a failing grade stay on my grade sheet after I clear the course?",
     "gold": ["An F and X grade obtained in any course shall be reflected in the grade sheet"],
     "expected": "Yes. An F/X grade remains reflected in the grade sheet; only the new grade is used for the CGPA.",
     "why": "Bi-encoder has Sec 6.3(7) at rank 2; the cross-encoder prefers chunks about other grade types "
            "(S/X, Incomplete) and pushes it out of the top 3."},
    {"category": "Counter-example: re-ranking hurts",
     "query": "How long can I stay enrolled before I run out of time to finish my degree?",
     "gold": ["six years (12 semesters)"],
     "expected": "Six years (12 semesters) at most, including leave and repeated semesters.",
     "why": "Sec 8.1 (maximum six years) is at bi-encoder rank 3 but the cross-encoder ranks it 4th."},
    # ---- limitation: the gold chunk is not near the top of the candidate pool ----------------
    {"category": "Limitation: neither pipeline finds it",
     "query": "How many credits do I need in total to get my degree?",
     "gold": ["minimum number of credits for a B.Tech. program is 156", "156 credits required for completion",
              "156 credits, required for graduation"],
     "expected": "156 credits (including 2 credits each of SG and CW).",
     "why": "The question shares 'credits' with every chunk about course credit sizes and loads (Sec 4, 5.8). "
            "The 156-credit rule sits at bi-encoder ranks 6 and 10; re-ranking only reaches rank 7."},
]


def norm(s):
    return re.sub(r"\s+", " ", s).replace("’", "'").replace("‘", "'").lower()


def is_gold(chunk_id, gold):
    # A chunk is gold if it contains any of the gold phrases (verbatim text of the answering clause).
    return any(norm(g) in norm(chunks[chunk_id]["text"]) for g in gold)


def gold_rank(id_list, gold):
    # 1-based rank of the first gold chunk, or None if absent.
    return next((r for r, i in enumerate(id_list, 1) if is_gold(i, gold)), None)


JUDGE_SYSTEM = (
    "You grade answers to questions about a university's regulations. Compare the candidate answer with the "
    "reference answer. Reply CORRECT if the candidate's main conclusion agrees with the reference and it does not "
    "state anything that contradicts the reference; missing minor details is fine. Reply INCORRECT if it "
    "contradicts the reference, gives a different number/rule, or says the context does not answer the question. "
    "Reply with exactly one word: CORRECT or INCORRECT.")


def judge(query, expected, answer):
    # Reference-based grading by the large local Qwen model (None if the llama.cpp server is not running).
    # The verdicts were spot-checked by hand; the judge is also one of the generators, so treat it as a guide.
    if "Qwen3.8-27B" not in LLMS:
        return None
    msgs = [{"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": f"Question: {query}\nReference answer: {expected}\n"
                                        f"Candidate answer: {answer}\nVerdict:"}]
    return LLMS["Qwen3.8-27B"](msgs)[0].strip().upper().startswith("CORRECT")


def rank_table(res, gold):
    # Top-10 table: bi-encoder rank/score vs cross-encoder rank/score for every candidate.
    ce_rank = {cid: r for r, cid in enumerate(res["rr_ids"], 1)}
    ce_score = dict(zip(res["rr_ids"], res["ce_scores"]))
    rows = []
    for r, (cid, s) in enumerate(zip(res["cand_ids"], res["bi_scores"]), 1):
        rows.append({"chunk": cid, "section": chunks[cid]["sec"], "starts with": chunks[cid]["text"][:60],
                     "bi_rank": r, "cosine": round(s, 3), "ce_rank": ce_rank[cid],
                     "ce_score": round(ce_score[cid], 2), "shift": r - ce_rank[cid],
                     "GOLD": "★" if is_gold(cid, gold) else ""})
    return pd.DataFrame(rows).sort_values("ce_rank").reset_index(drop=True)

In [9]:
failure_results = []
for fq in FAILURE_QUERIES:
    res = run_both_retrievals(fq["query"])
    b_rank, r_rank = gold_rank(res["cand_ids"], fq["gold"]), gold_rank(res["rr_ids"], fq["gold"])
    gold_chunk = next((i for i in res["cand_ids"] if is_gold(i, fq["gold"])), None)
    out = {**fq, "gold_chunk": gold_chunk, "baseline_rank": b_rank, "rerank_rank": r_rank,
           "baseline_top3": [chunks[i]["sec"] for i in res["baseline_top3"]],
           "rerank_top3": [chunks[i]["sec"] for i in res["rerank_top3"]],
           "table": rank_table(res, fq["gold"]), "answers": {}}
    for name in LLMS:
        a1, t1, n1 = generate_answer(name, fq["query"], res["baseline_top3"])
        a2, t2, n2 = generate_answer(name, fq["query"], res["rerank_top3"])
        out["answers"][name] = {"answer1": a1, "answer2": a2, "s1": t1, "s2": t2, "tok1": n1, "tok2": n2,
                                "ok1": judge(fq["query"], fq["expected"], a1), "ok2": judge(fq["query"], fq["expected"], a2)}
    failure_results.append(out)

    display(Markdown(f"### [{fq['category']}]  \n**Q:** {fq['query']}  \n"
                     f"**Gold:** \"{fq['gold'][0]}\" → chunk {gold_chunk} (Sec {chunks[gold_chunk]['sec'] if gold_chunk is not None else '-'})  \n"
                     f"**Rank shift:** #{b_rank} (bi-encoder) → **#{r_rank}** (cross-encoder)  \n"
                     f"*Why it is hard:* {fq['why']}"))
    display(out["table"])
    txt = (f"Baseline top-3 sections: `{out['baseline_top3']}` | "
           f"Re-ranked top-3 sections: `{out['rerank_top3']}`\n\n")
    mark = lambda ok: {True: "✅ correct", False: "❌ incorrect", None: "not judged"}[ok]
    txt += f"*Reference answer:* {fq['expected']}\n\n"
    for name, a in out["answers"].items():
        txt += (f"**{name}, Answer 1** ({mark(a['ok1'])}):\n\n> {a['answer1']}\n\n"
                f"**{name}, Answer 2** ({mark(a['ok2'])}):\n\n> {a['answer2']}\n\n")
    display(Markdown(txt + "---"))

### [Keyword trap / distractor]  
**Q:** My CGPA is 7.5. Can I take an extra course in my 5th semester?  
**Gold:** "overload only in one semester in the final year" → chunk 31 (Sec 5.8)  
**Rank shift:** #9 (bi-encoder) → **#1** (cross-encoder)  
*Why it is hard:* 'Extra course' and 'CGPA' match Sec 6.8 / 6.9 (extra credits beyond 156, extra courses to improve CGPA) and the Change History. The real rule is the OVERLOAD clause in Sec 5.8 (CGPA > 8.0, otherwise only in the final year), which never says 'extra course'.

,chunk,section,starts with,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,31,5.8,When a student registers for more credits than the stipulate,9,0.551,1,2.73,8,★
1,60,6.8,• CGPA computation for Honors students at the end of 8th sem,2,0.606,2,2.47,0,
2,61,6.9,"A B.Tech. student, who has already completed his/her graduat",1,0.619,3,1.91,-2,
3,58,6.8,Method: The below method will automatically decide which ext,6,0.594,4,1.81,2,
4,59,6.8,- CGPA will be computed based on the remaining credits. It i,7,0.565,5,1.80,2,
5,57,6.8,"B.Tech. students are allowed to do “extra credits"" beyond 15",5,0.602,6,-1.70,-1,
6,51,6.5,(ii) UG students to repeat (Improvement) of a particular cou,8,0.555,7,-1.80,1,
7,93,CH,(vi) Allowed to repeat a course completed with a pass grade\n,3,0.605,8,-2.73,-5,
8,94,CH,"Version 10 (July, 2019)\n(i) All operational guidelines are a",4,0.604,9,-4.14,-5,
9,92,CH,(ii) Registration of Online courses\n(iii) CGPA calculation\n(,10,0.543,10,-4.54,0,


Baseline top-3 sections: `['6.9', '6.8', 'CH']` | Re-ranked top-3 sections: `['5.8', '6.8', '6.9']`

*Reference answer:* No. An overload needs a CGPA of more than 8.0; a student without that may overload only in one semester of the final year (7th or 8th), so not in the 5th semester.

**Qwen2.5-1.5B, Answer 1** (❌ incorrect):

> No, according to Section 6.9 of the regulations, a B.Tech. student who has already completed their graduation requirements cannot register for more than two extra courses to improve their CGPA. The maximum number of additional courses they can take is limited to those registered within one extra semester after completing 156 credits.

**Qwen2.5-1.5B, Answer 2** (✅ correct):

> No, according to the regulations provided, second-year students with a CGPA of more than 8.0 can take an overload of up to 2 additional credits over 20 credits. Since your CGPA is 7.5, which is below the threshold for taking an overload, you cannot take an extra course in your 5th semester.

**Qwen3.8-27B, Answer 1** (❌ incorrect):

> The provided context does not answer this.

**Qwen3.8-27B, Answer 2** (✅ correct):

> According to Section 5.8, students with a CGPA of 7.5 are not eligible for overload in the 5th semester, as this option is restricted to the final year (7th or 8th semester) for those without a CGPA above 8.0. Therefore, you cannot take an extra course in your 5th semester.

---

### [Keyword trap / distractor]  
**Q:** After finishing 156 credits, for how many extra semesters can I stay to improve my CGPA?  
**Gold:** "will be allowed in only one extra semester" → chunk 61 (Sec 6.9)  
**Rank shift:** #7 (bi-encoder) → **#1** (cross-encoder)  
*Why it is hard:* '156 credits' and 'CGPA' are all over Sec 6.8 (the extra-credit CGPA method, four chunks). The answer ('only one extra semester') is in Sec 6.9.

,chunk,section,starts with,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,61,6.9,"A B.Tech. student, who has already completed his/her graduat",7,0.641,1,6.41,6,★
1,60,6.8,• CGPA computation for Honors students at the end of 8th sem,2,0.704,2,1.95,0,
2,57,6.8,"B.Tech. students are allowed to do “extra credits"" beyond 15",1,0.733,3,1.88,-2,
3,58,6.8,Method: The below method will automatically decide which ext,3,0.701,4,1.54,-1,
4,93,CH,(vi) Allowed to repeat a course completed with a pass grade\n,5,0.670,5,1.21,0,
5,59,6.8,- CGPA will be computed based on the remaining credits. It i,4,0.673,6,1.17,-2,
6,31,5.8,When a student registers for more credits than the stipulate,8,0.610,7,0.32,1,
7,92,CH,(ii) Registration of Online courses\n(iii) CGPA calculation\n(,9,0.603,8,-3.66,1,
8,94,CH,"Version 10 (July, 2019)\n(i) All operational guidelines are a",6,0.647,9,-4.99,-3,
9,96,CH,(ii) Grade improvement in summer courses. Applicable from 20,10,0.564,10,-6.81,0,


...

,chunk,section,starts with,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,70,7.4,Any circumstance leading to more than four semester leave/dr,1,0.607,1,-1.50,0,
1,40,6.2,"For medical reasons, only two medical applications including",10,0.475,2,-3.17,8,
2,68,7.4,(1) A student is expected to attend all classes. For any abs,6,0.498,3,-3.31,3,
3,80,8.1,The minimum duration for completing the B.Tech. program is 8,3,0.512,4,-4.48,-1,★
4,71,7.4,(3) If a student is absent from the Institute for more than,9,0.482,5,-5.14,4,
5,85,8.5,June 21 for the students who finish their requirements durin,2,0.515,6,-5.63,-4,
6,5,2,c) Summer Term. Starts around last week of May and ends arou,8,0.486,7,-5.69,1,
7,62,7,"For continuing in the undergraduate program, the student has",7,0.490,8,-6.38,-1,
8,69,7.4,"(2) A student may be allowed to take semester leave, with pe",5,0.501,9,-6.44,-4,
9,63,7.1,Rules for the first year (excluding the summer term schedule,4,0.509,10,-6.47,-6,


Baseline top-3 sections: `['7.4', '8.5', '8.1']` | Re-ranked top-3 sections: `['7.4', '6.2', '7.4']`

*Reference answer:* Six years (12 semesters) at most, including leave and repeated semesters.

**Qwen2.5-1.5B, Answer 1** (❌ incorrect):

> According to Section 8.1 of the regulations, the minimum duration for completing the B.Tech. program is 8 semesters, excluding any semester leaves or repeated semesters. Therefore, you have up to 8 semesters from your first registration date to complete all requirements before running out of time to finish your degree.

**Qwen2.5-1.5B, Answer 2** (❌ incorrect):

> According to Section 7.4 of the regulations, a student must necessarily be registered and physically present on campus for 8 semesters during their degree program, except for the last semester where they can go for internship if eligible. Therefore, a student can stay enrolled for up to 8 semesters before running out of time to complete their degree.

**Qwen3.8-27B, Answer 1** (✅ correct):

> The maximum duration within which you must complete the B.Tech. program requirements is six years, which corresponds to 12 semesters. This timeframe includes any semesters in which you took a semester leave or repeated a semester or year. [Section 8.1]

**Qwen3.8-27B, Answer 2** (❌ incorrect):

> According to Section 7.4, a student must be registered and physically present on campus for 8 semesters during the degree program. Any circumstance leading to more than four semester leaves or drops may result in the automatic termination of the program.

---

### [Limitation: neither pipeline finds it]  
**Q:** How many credits do I need in total to get my degree?  
**Gold:** "minimum number of credits for a B.Tech. program is 156" → chunk 57 (Sec 6.8)  
**Rank shift:** #6 (bi-encoder) → **#7** (cross-encoder)  
*Why it is hard:* The question shares 'credits' with every chunk about course credit sizes and loads (Sec 4, 5.8). The 156-credit rule sits at bi-encoder ranks 6 and 10; re-ranking only reaches rank 7.

,chunk,section,starts with,bi_rank,cosine,ce_rank,ce_score,shift,GOLD
0,30,5.8,"In general, students will take courses as per the program. N",1,0.601,1,1.06,0,
1,35,5.8,(3) Must have completed all core courses and other graduatio,7,0.534,2,0.25,5,
2,14,4,"(1) Every semester the Institute offers some courses, taught",2,0.589,3,-0.83,-1,
3,15,4,b) A 2-credit course. The course will have 1.5 hours of lect,4,0.550,4,-1.76,0,
4,33,5.8,Students can avail underload below 12 credits (as low as 4 c,9,0.524,5,-2.05,4,
5,34,5.8,(4) Must have been left with 4 credits of IP/IS/UR/Online co,8,0.528,6,-2.08,2,
6,80,8.1,The minimum duration for completing the B.Tech. program is 8,10,0.517,7,-2.25,3,★
7,16,4,c) A 1-credit course. The course is likely to be run as a sh,3,0.555,8,-2.42,-5,
8,57,6.8,"B.Tech. students are allowed to do “extra credits"" beyond 15",6,0.535,9,-2.76,-3,★
9,82,8.3,A student enrolled in a B.Tech. program may also graduate wi,5,0.543,10,-3.79,-5,


Baseline top-3 sections: `['5.8', '4', '4']` | Re-ranked top-3 sections: `['5.8', '5.8', '4']`

*Reference answer:* 156 credits (including 2 credits each of SG and CW).

**Qwen2.5-1.5B, Answer 1** (❌ incorrect):

> To determine the total number of credits needed for your degree, refer to Section 5.8 of the regulations. It states that "Normal load for the first- and second-year students is 16-20 credits" and "For the third- and fourth-year students is 16-22 credits." This information outlines the typical academic load required for different years of study at IIIT-Delhi.

**Qwen2.5-1.5B, Answer 2** (❌ incorrect):

> To get your degree, you typically need to complete between 16-22 credits depending on whether you're in the first or second year, respectively.

**Qwen3.8-27B, Answer 1** (❌ incorrect):

> The provided context does not answer this.

**Qwen3.8-27B, Answer 2** (❌ incorrect):

> The provided context does not answer this.

---

In [10]:
# Summary: rank shifts and whether each answer was judged correct
shift_df = pd.DataFrame([{
    "category": r["category"], "query": r["query"], "gold_chunk": r["gold_chunk"],
    "bi_rank": r["baseline_rank"], "ce_rank": r["rerank_rank"],
    "shift": f"Chunk {r['gold_chunk']}: #{r['baseline_rank']} → #{r['rerank_rank']}",
    "in_top3_baseline": (r["baseline_rank"] or 99) <= 3, "in_top3_rerank": (r["rerank_rank"] or 99) <= 3,
    **{f"{m} A1 correct": a["ok1"] for m, a in r["answers"].items()},
    **{f"{m} A2 correct": a["ok2"] for m, a in r["answers"].items()}}
    for r in failure_results])
shift_df

,category,query,gold_chunk,bi_rank,ce_rank,shift,in_top3_baseline,in_top3_rerank,Qwen2.5-1.5B A1 correct,Qwen3.8-27B A1 correct,Qwen2.5-1.5B A2 correct,Qwen3.8-27B A2 correct
0,Keyword trap / distractor,My CGPA is 7.5. Can I take an extra course in my 5th semester?,31,9,1,Chunk 31: #9 → #1,False,True,False,False,True,True
1,Keyword trap / distractor,"After finishing 156 credits, for how many extra semesters can I stay to improve my CGPA?",61,7,1,Chunk 61: #7 → #1,False,True,False,False,True,True
2,Keyword trap / synonyms,Can I take a gap semester to work on my startup?,69,4,2,Chunk 69: #4 → #2,False,True,False,False,True,True
3,Keyword trap / synonyms,Can I spend a year studying at a foreign university?,55,6,2,Chunk 55: #6 → #2,False,True,False,False,True,True
4,Exception / corner case,Does the six-year maximum duration include semester leaves?,80,6,1,Chunk 80: #6 → #1,False,True,False,False,True,True
5,Exception / corner case,Does an online course count as an in-class course during my internship semester?,36,4,1,Chunk 36: #4 → #1,False,True,True,True,True,True
6,Negation & constraints,How many semester leaves are allowed in the whole program?,70,4,1,Chunk 70: #4 → #1,False,True,False,False,True,True
7,Negation & constraints,Is a student expelled from another institute on disciplinary grounds eligible for admission?,8,9,1,Chunk 8: #9 → #1,False,True,True,False,True,True
8,Other: vital chunk buried at rank 4-10,When is a student's program terminated for poor academic performance?,67,6,3,Chunk 67: #6 → #3,False,True,False,False,True,True
9,Counter-example: re-ranking hurts,Does a failing grade stay on my grade sheet after I clear the course?,48,2,4,Chunk 48: #2 → #4,True,False,False,True,False,False


In [11]:
# Closed-book on the same stress-test questions: the same models with NO context at all
for r in failure_results:
    r["closed_book"] = {}
    for name, llm in LLMS.items():
        ans = llm([{"role": "system", "content": CLOSED_BOOK_SYSTEM}, {"role": "user", "content": r["query"]}])[0]
        r["closed_book"][name] = {"answer": ans, "correct": judge(r["query"], r["expected"], ans)}

answer_quality = pd.DataFrame({
    name: {"closed-book (no context)": sum(bool(r["closed_book"][name]["correct"]) for r in failure_results),
           "Answer 1 (baseline top-3)": sum(bool(r["answers"][name]["ok1"]) for r in failure_results),
           "Answer 2 (re-ranked top-3)": sum(bool(r["answers"][name]["ok2"]) for r in failure_results)}
    for name in LLMS})
print(f"Correct answers out of the {len(failure_results)} stress-test questions (reference-based judge):")
display(answer_quality)
with pd.option_context("display.max_colwidth", 160):
    display(pd.DataFrame([{"query": r["query"], **{f"{m} (closed-book)": r["closed_book"][m]["answer"]
                           for m in LLMS}} for r in failure_results]))

Correct answers out of the 12 stress-test questions (reference-based judge):


,Qwen2.5-1.5B,Qwen3.8-27B
closed-book (no context),0,1
Answer 1 (baseline top-3),2,3
Answer 2 (re-ranked top-3),9,9


,query,Qwen2.5-1.5B (closed-book),Qwen3.8-27B (closed-book)
0,My CGPA is 7.5. Can I take an extra course in my 5th semester?,"Yes, students can typically take up to two additional courses per semester at IIIT Delhi.","Yes, you can take an extra course in your 5th semester, as IIIT-Delhi typically allows students with a CGPA of 7.5 or higher to register for additional cour..."
1,"After finishing 156 credits, for how many extra semesters can I stay to improve my CGPA?",The duration of staying at IIIT Delhi after completing 156 credits is not specified in the given information.,"You can stay for up to 2 extra semesters (4 semesters total beyond the standard 8) to complete your degree and improve your CGPA, provided you have not alre..."
2,Can I take a gap semester to work on my startup?,"Yes, students can typically take up to two semesters (6 months) of study leave for internships or entrepreneurship projects like startups.","No, you cannot take a gap semester to work on a startup; IIIT-Delhi's B.Tech regulations do not allow students to drop out or take a leave of absence for su..."
3,Can I spend a year studying at a foreign university?,"Yes, it is possible to study for one academic year abroad with proper planning and funding arrangements. Many students choose to do so as part of their educ...","Yes, IIIT-Delhi offers a Semester Abroad program that allows students to spend one semester (typically 4-5 months) at a partner foreign university, rather t..."
4,Does the six-year maximum duration include semester leaves?,"Yes, the six-year maximum duration for a Bachelor of Technology (B.Tech) program at IIIT-Delhi includes semester breaks or leaves. Students are allowed to t...","No, the six-year maximum duration does not include semester leaves; the maximum period is extended by the number of semesters for which a leave of absence i..."
5,Does an online course count as an in-class course during my internship semester?,"No, an online course does not count as an in-class course for internships. Internship credits are typically earned through on-site work experience under dir...","No, online courses generally do not count as in-class courses for internship eligibility at IIIT-Delhi, as internships typically require you to be enrolled ..."
6,How many semester leaves are allowed in the whole program?,"The entire duration of a Bachelor of Technology (B.Tech) program at IIIT Delhi is divided into four semesters, each lasting for approximately 15 weeks. Ther...",IIIT-Delhi allows a maximum of **2** semester leaves (also known as breaks) during the entire B.Tech program.
7,Is a student expelled from another institute on disciplinary grounds eligible for admission?,"Yes, students expelled from other institutes are generally not eligible to apply directly to IIIT Delhi unless they have completed their studies elsewhere a...","No, a student expelled from another institute on disciplinary grounds is not eligible for admission to IIIT-Delhi."
8,When is a student's program terminated for poor academic performance?,A student's program at IIIT-Delhi can be terminated for poor academic performance if they fail to meet the minimum CGPA requirement or complete their course...,A student's program is terminated if they fail to maintain a minimum Cumulative Grade Point Average (CGPA) of 5.00 (on a 10-point scale) by the end of the 4...
9,Does a failing grade stay on my grade sheet after I clear the course?,"No, a failing grade does not typically remain on a student's transcript or grade sheet even after passing the course.","No, a failing grade does not stay on your grade sheet after you clear the course; the final grade recorded is the one from the successful attempt."


## 7. Aggregate retrieval evaluation (labelled query set)

To check that the stress tests aren't cherry-picked, I score both pipelines on **151 labelled questions**, each with
gold phrase(s). They come in three styles:

* **rule-phrased** (80): questions worded close to the regulation text ("How many credits can be transferred from another university?")
* **student-phrased** (50): how a student would actually ask, with informal words and synonyms ("MOOC", "kicked out", "gap semester", "endsem")
* **exception** (21): questions whose answer is an exception, a restriction or an inclusion/exclusion detail

Metrics:

* **Hit@1 / Hit@3**: is a gold chunk at rank 1 / in the top 3 (top 3 is what the LLM sees)?
* **MRR@10**: mean reciprocal rank of the first gold chunk within the 10 candidates.
* **Recall@10**: is a gold chunk in the candidate pool at all? (This is the ceiling for re-ranking: the
  cross-encoder can only reorder what the bi-encoder retrieved.)

In [12]:
EVAL_SET = [
 ('rule-phrased', 'How many credits are required to graduate with a B.Tech. degree?',
    ['minimum number of credits for a B.Tech. program is 156', '156 credits required for completion', '156 credits, required for graduation']),
 ('rule-phrased', 'Can a first-year student take extra credits beyond the normal load?',
    ['First year students cannot take any overload']),
 ('rule-phrased', 'What CGPA does a second-year student need to take an overload?',
    ['Second year students having a CGPA of more than 8.0']),
 ('rule-phrased', 'How many grade points does an F grade carry?',
    ['F (Fail) 2']),
 ('rule-phrased', 'Is the F grade included when computing CGPA?',
    ['F Grade is not counted in the calculation of CGPA', "'F' grade shall not be counted in the calculation of CGPA", '‘F’ grade shall not be counted in the calculation of CGPA']),
 ('rule-phrased', 'What happens to an Incomplete grade that is not converted in time?',
    ['automatically converted to F grade']),
 ('rule-phrased', 'How many online course credits can count towards the B.Tech. degree?',
    ['upto 8 credits of Online Courses']),
 ('rule-phrased', 'Can an online course be registered as an independent project?',
    ['and not in the category of Independent Project']),
 ('rule-phrased', 'How many credits can a student earn for a TAship in one semester?',
    ['maximum of 2 credits for TAship']),
 ('rule-phrased', 'Do TA credits count separately from online course credits?',
    ['counted within the approved bucket of 8 credits']),
 ('rule-phrased', 'How many credits can I take in the summer term?',
    ['up to 6 credits of courses that are offered during the summer term']),
 ('rule-phrased', 'Can a graduating student take more than 6 credits in the summer?',
    ['graduating batch, who may be allowed to take up to 8 credits']),
 ('rule-phrased', 'Can I add a course after registering in the summer term?',
    ['shall not be allowed to add a course after registration']),
 ('rule-phrased', 'Is late registration allowed for the summer term?',
    ['There will not be any late registration in the summer term']),
 ('rule-phrased', 'How many courses can a student withdraw from after the add/drop deadline?',
    ['withdraw from at most one elective course']),
 ('rule-phrased', 'What happens if a student does not register on ERP for two semesters?',
    ['Absent Without Leave” for two semesters']),
 ('rule-phrased', 'What is the lowest number of credits a student can register for in the final semester?',
    ['as low as 4 credits']),
 ('rule-phrased', 'How many credits must be completed to apply for an internship at the end of semester 6?',
    ['Must have completed 126 credits']),
 ('rule-phrased', 'How many credits are needed to go for an internship after the 7th semester?',
    ['Must have completed 148 credits']),
 ('rule-phrased', 'Can a student on internship take classroom courses that semester?',
    ['not be allowed to do any in-class course during the internship']),
 ('rule-phrased', 'How many times can a student use medical reasons to get a make-up exam?',
    ['only two medical applications']),
 ('rule-phrased', 'Within how many days must a medical certificate be uploaded for a missed exam?',
    ['within five working days']),
 ('rule-phrased', 'When must a request for a make-up exam be made?',
    ['within two days of the date of the scheduled examination']),
 ('rule-phrased', 'Who can request a change to a grade that has already been awarded?',
    ['shall not be changed unless the request is made by the Instructor-in-Charge']),
 ('rule-phrased', 'How much can a grade improve after a refresher module?',
    ['at most half the letter grade improvement']),
 ('rule-phrased', 'Can a student with a B grade use a refresher module to improve it?',
    ['only students with F or D grade']),
 ('rule-phrased', 'If I repeat a course I already passed, which grade counts?',
    ['only the best Grade in that course will be counted']),
 ('rule-phrased', 'Can a repeated course be dropped late after the mid-semester exam?',
    ['not be permitted for late drop after mid-semester examination']),
 ('rule-phrased', 'What can a student do after failing a core course twice?',
    ['after getting an F twice in the course']),
 ('rule-phrased', 'Does a student have to repeat a failed elective course?',
    ['elective/optional course, the student can replace the course']),
 ('rule-phrased', 'How many credits can be transferred from another university?',
    ['maximum credit transfer allowed is 40 credits']),
 ('rule-phrased', 'Do grades earned at another university count toward my CGPA?',
    ['grades corresponding to the transferred credits will not be counted']),
 ('rule-phrased', 'How many credits of worst grades can be dropped from the CGPA by doing extra credits?',
    ['maximum limit of at most 8 credits', 'M= min {8, N}']),
 ('rule-phrased', 'What is the baseline credit count for CGPA computation at the end of the 7th semester?',
    ['baseline credits will be 136']),
 ('rule-phrased', 'Do extra credits change the CGPA computed in the first five semesters?',
    ['CGPA computation will remain the same for first 5 semesters']),
 ('rule-phrased', 'Can a student who has finished the graduation requirements register for more courses to raise the CGPA?',
    ['at most two extra courses to improve his/her CGPA']),
 ('rule-phrased', 'Are students who take extra courses after completing requirements eligible for medals?',
    ['will not be considered for any award/medal']),
 ('rule-phrased', 'How many failed courses in the first year force a student to repeat the year?',
    ['fails in any 5 or more courses']),
 ('rule-phrased', 'Which earlier credits can a student repeating the first year keep?',
    ['communication (COM101) and SSH courses']),
 ('rule-phrased', 'What SGPA puts a student on academic warning?',
    ['SGPA/CGPA of 4.5 or less']),
 ('rule-phrased', 'Does being placed on academic warning lead to termination?',
    ['however this will not lead to termination']),
 ('rule-phrased', 'Can a student on academic warning contest a student council election?',
    ['not be allowed to hold any elected or nominated post']),
 ('rule-phrased', 'Can a student on probation play for the institute team at an outside event?',
    ['not be allowed to participate in any event outside the Institute']),
 ('rule-phrased', "When is a student's program terminated for poor academic performance?",
    ['low SGPA (of <4.0) in two consecutive semesters']),
 ('rule-phrased', 'How many semester leaves are allowed in the whole program?',
    ['more than four semester leave']),
 ('rule-phrased', 'Can semester leave be taken after the add/drop deadline?',
    ['No semester leave will be granted after the last date of add/drop']),
 ('rule-phrased', 'Is the fee refunded if a registered semester is converted into a semester leave?',
    ['no fee refund is admissible']),
 ('rule-phrased', 'What happens if a student is absent for more than 20 days in a semester?',
    ['more than 20 days in a semester']),
 ('rule-phrased', 'Does absence while representing the institute at an event count against the student?',
    ['absence due to this participation will not be counted']),
 ('rule-phrased', 'Within how long must a terminated student appeal?',
    ['appeals shall be made to the Senate within two weeks']),
 ('rule-phrased', 'When can a student change their branch?',
    ['only after the grades of the first semester are out']),
 ('rule-phrased', 'What grades are needed to transfer into CSE?',
    ['B or better grade in Introduction to Programming']),
 ('rule-phrased', 'What are the requirements to transfer into the ECE program?',
    ['better grade in both Digital Circuits and Mathematics-I']),
 ('rule-phrased', "By how much can a program's strength change because of branch transfers?",
    ['shall not reduce by more than 10%']),
 ('rule-phrased', 'How are ties in CGPA broken for branch transfer?',
    ['Tie Breaking Rule I']),
 ('rule-phrased', 'When can a student migrate to the dual degree program?',
    ['immediately after the declaration of Semester 6 Results']),
 ('rule-phrased', 'Can a student leave the dual degree after using the placement service?',
    ['provided he/she has not availed the Placement Service']),
 ('rule-phrased', 'What is the maximum time allowed to finish the B.Tech.?',
    ['six years (12 semesters)']),
 ('rule-phrased', 'What CGPA is needed to graduate with honors?',
    ['CGPA of 8.0 or more at graduation']),
 ('rule-phrased', 'How many extra credits are needed for an honors degree?',
    ['at least 12 extra discipline credits']),
 ('rule-phrased', 'What date is printed on the degree for students who finish in the summer term?',
    ['September 21 for those who finish']),
 ('rule-phrased', 'When is convocation usually held?',
    ['second half of October']),
 ('rule-phrased', 'What are the minimum class XII marks to be eligible for admission?',
    ['70% or more marks in the aggregate']),
 ('rule-phrased', 'What percentage of seats is reserved for Delhi candidates?',
    ['85% of the seats are reserved']),
 ('rule-phrased', 'Can credits earned as a non-degree student be used for an IIIT-Delhi degree later?',
    ['cannot be applied towards any degree']),
 ('rule-phrased', 'Is a student expelled from another institute on disciplinary grounds eligible for admission?',
    ['on disciplinary grounds are not eligible']),
 ('rule-phrased', 'What is the normal credit load for a third-year student?',
    ['third- and fourth-year students is 16-22 credits']),
 ('rule-phrased', 'How far below 16 credits can a student go with an underload?',
    ['maximum of 4 credits below 16 credits']),
 ('rule-phrased', 'Are students on academic warning allowed to take an overload?',
    ['Students under academic warning may not be permitted any overload', 'No academic overload shall be permitted']),
 ('rule-phrased', 'What grades are given for online courses?',
    ['only S or X grade is awarded']),
 ('rule-phrased', 'How long is the summer term compared to a regular semester?',
    ['about half the duration of the regular semesters']),
 ('rule-phrased', 'How many hours of lectures does a 1-credit course have?',
    ['about 10 hours of lecture']),
 ('rule-phrased', 'Can a student whose CGPA is not above 8 ever take an overload?',
    ['overload only in one semester in the final year']),
 ('rule-phrased', 'What is the deadline for converting an I grade?',
    ['converted to a regular grade within six weeks', 'must be converted to one of the letter grades']),
 ('rule-phrased', 'Can admission be cancelled after the student has already joined?',
    ['cancelled by the Senate even at a later date']),
 ('rule-phrased', 'What is the deadline for a provisionally admitted student to show proof of eligibility?',
    ['following September 30th']),
 ('rule-phrased', 'Can a degree already awarded be taken back?',
    ['withdraw an already awarded degree']),
 ('rule-phrased', 'What penalties can be given for academic dishonesty?',
    ['semester/year/two year drop or even expulsion']),
 ('rule-phrased', 'Is pre-registration required for first-semester students?',
    ['academic registration for non-first semester students']),
 ('rule-phrased', 'Is an F grade counted in the SGPA?',
    ['it is counted in the calculation of SGPA']),
 ('student-phrased', 'How many credits do I need in total to get my degree?',
    ['minimum number of credits for a B.Tech. program is 156', '156 credits required for completion', '156 credits, required for graduation']),
 ('student-phrased', "What's the minimum CGPA to avoid getting kicked out of the program?",
    ['low SGPA (of <4.0) in two consecutive semesters']),
 ('student-phrased', 'Is the summer term compulsory?',
    ['Registering in the summer term is optional']),
 ('student-phrased', 'Is there a cap on how many MOOC credits I can count towards graduation?',
    ['upto 8 credits of Online Courses']),
 ('student-phrased', 'Can I get academic credit for being a teaching assistant?',
    ['maximum of 2 credits for TAship']),
 ('student-phrased', 'How long can I stay enrolled before I run out of time to finish my degree?',
    ['six years (12 semesters)']),
 ('student-phrased', 'Can I take a gap semester to work on my startup?',
    ['entrepreneurship']),
 ('student-phrased', 'I fell sick and missed my endsem. What can I do?',
    ['Make-up Exam: If a student, for valid reasons such as illness']),
 ('student-phrased', 'My CGPA is 7.5. Can I take an extra course in my 5th semester?',
    ['overload only in one semester in the final year']),
 ('student-phrased', 'Can I switch from ECE to CSE in my second year?',
    ['This is the only time branch change will occur']),
 ('student-phrased', 'Will grades from my exchange semester show up in my CGPA?',
    ['grades corresponding to the transferred credits will not be counted']),
 ('student-phrased', "If I failed a course and passed it the next semester, which semester's SGPA does the pass grade go into?",
    ['count towards the SGPA of the semester in which the course is passed']),
 ('student-phrased', 'Does a failing grade stay on my grade sheet after I clear the course?',
    ['An F and X grade obtained in any course shall be reflected in the grade sheet']),
 ('student-phrased', 'Who decides situations that the regulations do not cover?',
    ['not covered in the regulations shall be referred to the Academic Affairs committee']),
 ('student-phrased', 'Can I get some proof of my degree before the convocation ceremony?',
    ['issue a provisional certificate']),
 ('student-phrased', 'If I finish all requirements in the monsoon semester, what date goes on my degree?',
    ['January 21 for those who finish']),
 ('student-phrased', 'Is there a penalty for not filling in my course preferences on time?',
    ['fails to do pre-registration of courses, a penalty may be imposed']),
 ('student-phrased', 'Can I drop a course during the summer term?',
    ['During the summer term, a student may be permitted to drop one course']),
 ('student-phrased', 'What does a W on my transcript mean?',
    ['“W” grade shall be printed in the transcript', 'grade shall be printed in the transcript of the students who do ‘Late Drop’']),
 ('student-phrased', 'Do audit courses affect my CGPA?',
    ['They are not incorporated in SGPA/CGPA calculation']),
 ('student-phrased', 'Which students are barred from all awards and medals?',
    ['will not be considered for any award/medal']),
 ('student-phrased', 'Is attendance in classes mandatory?',
    ['A student is expected to attend all classes']),
 ('student-phrased', 'Can I transfer to CSE if I got a B- in Intro to Programming?',
    ['B or better grade in Introduction to Programming']),
 ('student-phrased', 'What happens to a first-year repeater who keeps failing?',
    ['the termination clause (7.3 clause of UG regulation) will be applicable']),
 ('student-phrased', 'If I took a semester off in first year and then failed five courses, do I still have to repeat the year?',
    ['provided he/she did not take semester leave']),
 ('student-phrased', 'Do I have to physically be on campus for all eight semesters?',
    ['physically be present on campus for 8 semesters']),
 ('student-phrased', 'How soon after first-semester results must I ask for a branch change?',
    ['within a week from the date of announcement of results']),
 ('student-phrased', 'Can I get an incomplete in a regular course for a non-medical reason?',
    ['incomplete grade “I” is meant primarily for medical reasons']),
 ('student-phrased', 'After finishing 156 credits, for how many extra semesters can I stay to improve my CGPA?',
    ['will be allowed in only one extra semester']),
 ('student-phrased', 'When is the late registration fee waived?',
    ['late fee may be waived if the delay is due to an academic activity']),
 ('student-phrased', 'Can the institute change the number of seats after the merit list is out?',
    ['No change in the total number of seats will be done after the merit list']),
 ('student-phrased', 'Can the Senate change these regulations?',
    ['the Senate has the right to modify']),
 ('student-phrased', 'Can a student take fewer courses than usual in a semester?',
    ['it is called the underload']),
 ('student-phrased', "What happens if I don't register at all for a semester?",
    ['Failure to register may lead to termination']),
 ('student-phrased', 'How are grade points weighted when computing my semester average?',
    ['SGPA = (u1.w1 + u2.w2']),
 ('student-phrased', "If my CGPA ties with someone else's for a branch change, what decides it?",
    ['Tie Breaking Rule I']),
 ('student-phrased', 'Can I go back from the dual degree to the normal B.Tech.?',
    ['may opt out from migration before the add/drop date of his/her 7th semester']),
 ('student-phrased', 'Who signs the degree scroll of students who finish in January?',
    ['scroll of such students may be signed by the Chairman, BoG']),
 ('student-phrased', "What happens if I lie on my admission form and it's found out in my third year?",
    ['cancelled by the Senate even at a later date']),
 ('student-phrased', 'Who checks my medical certificate when I ask for a make-up exam?',
    ['get the medical certificate verified from the Institute doctor']),
 ('student-phrased', "Will I get a fail grade if I don't finish my incomplete course work?",
    ['automatically converted to F grade']),
 ('student-phrased', 'Can I spend a year studying at a foreign university?',
    ['spend up to 2 semesters at another University']),
 ('student-phrased', 'What is the most credits I can transfer in a single semester abroad?',
    ['maximum of 20 credits in a semester']),
 ('student-phrased', 'Who handles cheating cases?',
    ['Disciplinary Action Committee (DAC)']),
 ('student-phrased', 'Can a student on academic probation be a hostel secretary?',
    ['not be allowed to hold any elected or nominated post']),
 ('student-phrased', 'Does my parent need to sign anything if I am put on academic warning?',
    ['countersigned by the parent/guardian']),
 ('student-phrased', 'How long is the end-semester exam period?',
    ['last 10 days in the semester are reserved for the end-semester examination']),
 ('student-phrased', 'Can a student from another university study here without getting an IIIT-Delhi degree?',
    ['A non-degree student is a student registered for a degree']),
 ('student-phrased', 'Is there extra credit for students who take lots of courses?',
    ['allowed to do “extra credits" beyond 156 credits', 'allowed worst grades in N credits']),
 ('student-phrased', 'What exactly counts as having a backlog?',
    ['A student has a backlog if he/she has not taken or has taken but failed']),
 ('exception', 'Can I take more than 6 credits in the summer before my final semester?',
    ['graduating batch, who may be allowed to take up to 8 credits']),
 ('exception', "I'm in my final year and my CGPA is only 7. Can I register for 24 credits?",
    ['overload only in one semester in the final year']),
 ('exception', 'Does time away representing the college at a hackathon count toward the 20-day absence limit?',
    ['absence due to this participation will not be counted']),
 ('exception', 'Can a student who is repeating the first year still change branch?',
    ['Repeaters who are repeating first year are also allowed branch transfer']),
 ('exception', 'Can I take semester leave after add/drop if I get seriously ill?',
    ['other than for the critical reasons such as ill health']),
 ('exception', 'Can I replace a failed core course with an elective?',
    ['after getting an F twice in the course']),
 ('exception', 'Can I go down to 8 credits in my 6th semester?',
    ['as low as 4 credits']),
 ('exception', 'Are grades from a partner university ever counted in my CGPA?',
    ['if there is an MoU with the University']),
 ('exception', 'Can someone with less than 70% in class XII apply?',
    ['Relaxation is given for category students']),
 ('exception', 'Can I add an internship without taking semester leave?',
    ['without taking semester leave can do so under this clause']),
 ('exception', 'Is an F grade ignored when computing the semester GPA too?',
    ['it is counted in the calculation of SGPA']),
 ('exception', 'Can a student with a C grade get a better grade through the vacation refresher module?',
    ['only students with F or D grade']),
 ('exception', 'Can I attend the refresher module even if I passed the course with an A?',
    ['although any student can attend']),
 ('exception', 'Can a TAship count as part of my normal course load?',
    ['load of TAship will be considered equivalent to that of a 4 credit course']),
 ('exception', 'Can a 2-credit course be completed in one week?',
    ['no less than 3 weeks']),
 ('exception', 'Does an online course count as an in-class course during my internship semester?',
    ['register for 4 credits of only IP/IS/UR/BTP/OC courses']),
 ('exception', 'If I miss a mid-semester exam for medical reasons, does it count toward the medical make-up limit?',
    ['including mid-semester and end-semester']),
 ('exception', 'Can my program be terminated if I am on academic warning only once?',
    ['however this will not lead to termination']),
 ('exception', 'Can a student who did not take semester leave be terminated for taking too many leaves?',
    ['more than four semester leave/drop may result in the automatic termination']),
 ('exception', 'Does the minimum 8-semester duration include semesters spent on leave?',
    ['excluding the semesters in which the student took semester leave or the repeated']),
 ('exception', 'Does the six-year maximum duration include semester leaves?',
    ['including the semester in which student took semester leave']),
]

missing = [q for _, q, g in EVAL_SET if not any(is_gold(i, g) for i in range(len(chunks)))]
assert not missing, f"gold phrase not found in any chunk: {missing}"

rows = []
for style, q, gold in EVAL_SET:
    res = run_both_retrievals(q)
    rows.append({"style": style, "query": q, "gold": gold[0],
                 "bi_rank": gold_rank(res["cand_ids"], gold), "ce_rank": gold_rank(res["rr_ids"], gold)})
eval_df = pd.DataFrame(rows)


def metrics(r):
    return {"Hit@1": (r == 1).mean(), "Hit@3": (r <= 3).mean(),
            "MRR@10": r.apply(lambda x: 1 / x if pd.notna(x) else 0).mean(),
            "Recall@10": r.notna().mean()}


metrics_df = pd.DataFrame({"Pipeline 1 (bi-encoder)": metrics(eval_df.bi_rank),
                           "Pipeline 2 (re-ranked)": metrics(eval_df.ce_rank)}).T.round(3)
print(f"All {len(eval_df)} labelled queries")
display(metrics_df)

by_style = pd.concat({style: pd.DataFrame({"Pipeline 1": metrics(g.bi_rank), "Pipeline 2": metrics(g.ce_rank)}).T
                      for style, g in eval_df.groupby("style", sort=False)}).round(3)
by_style["n"] = [len(eval_df[eval_df["style"] == s]) for s, _ in by_style.index]
print("By query style")
display(by_style)

All 151 labelled queries


,Hit@1,Hit@3,MRR@10,Recall@10
Pipeline 1 (bi-encoder),0.656,0.854,0.768,0.974
Pipeline 2 (re-ranked),0.854,0.907,0.890,0.974


By query style


Hit@1  Hit@3  MRR@10  Recall@10   n
rule-phrased    Pipeline 1  0.775  0.950   0.864       1.00  80
                Pipeline 2  0.975  0.988   0.982       1.00  80
student-phrased Pipeline 1  0.580  0.720   0.677       0.92  50
                Pipeline 2  0.680  0.760   0.749       0.92  50
exception       Pipeline 1  0.381  0.810   0.619       1.00  21
                Pipeline 2  0.810  0.952   0.877       1.00  21

In [13]:
eval_df["effect"] = np.select(
    [(eval_df.bi_rank.fillna(99) > 3) & (eval_df.ce_rank.fillna(99) <= 3),
     (eval_df.bi_rank.fillna(99) <= 3) & (eval_df.ce_rank.fillna(99) > 3)],
    ["rescued into top-3", "pushed out of top-3"], default="no top-3 change")
display(eval_df["effect"].value_counts().to_frame("queries"))
eval_df[eval_df.effect != "no top-3 change"].sort_values(["effect", "style"])

,queries
effect,
no top-3 change,137
rescued into top-3,11
pushed out of top-3,3


,style,query,gold,bi_rank,ce_rank,effect
148,exception,Can a student who did not take semester leave be terminated for taking too many leaves?,more than four semester leave/drop may result in the automatic termination,2.0,4.0,pushed out of top-3
85,student-phrased,How long can I stay enrolled before I run out of time to finish my degree?,six years (12 semesters),3.0,4.0,pushed out of top-3
92,student-phrased,Does a failing grade stay on my grade sheet after I clear the course?,An F and X grade obtained in any course shall be reflected in the grade sheet,2.0,4.0,pushed out of top-3
131,exception,I'm in my final year and my CGPA is only 7. Can I register for 24 credits?,overload only in one semester in the final year,8.0,1.0,rescued into top-3
145,exception,Does an online course count as an in-class course during my internship semester?,register for 4 credits of only IP/IS/UR/BTP/OC courses,4.0,1.0,rescued into top-3
149,exception,Does the minimum 8-semester duration include semesters spent on leave?,excluding the semesters in which the student took semester leave or the repeated,8.0,1.0,rescued into top-3
150,exception,Does the six-year maximum duration include semester leaves?,including the semester in which student took semester leave,6.0,1.0,rescued into top-3
43,rule-phrased,When is a student's program terminated for poor academic performance?,low SGPA (of <4.0) in two consecutive semesters,6.0,3.0,rescued into top-3
44,rule-phrased,How many semester leaves are allowed in the whole program?,more than four semester leave,4.0,1.0,rescued into top-3
65,rule-phrased,Is a student expelled from another institute on disciplinary grounds eligible for admission?,on disciplinary grounds are not eligible,9.0,1.0,rescued into top-3


## 8. Latency profiling

I time each stage separately (after a warm-up), repeating every query `N_REPEATS` times:

* **Pipeline 1 retrieval** = query embedding + FAISS top-10 search
* **Pipeline 2 retrieval** = Pipeline 1 retrieval + cross-encoder scoring of 10 pairs + sort
* **LLM generation** is timed separately per model (it is the same work for both pipelines)

In [14]:
N_REPEATS = 10
lat_queries = [q for _, q, _ in EVAL_SET]

for q in lat_queries[:5]:            # warm-up (kernel/JIT/cache effects)
    run_both_retrievals(q)

embed_t, search_t, ce_t = [], [], []
for _ in range(N_REPEATS):
    for q in lat_queries:
        t0 = time.perf_counter()
        qv = bi_encoder.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        t1 = time.perf_counter()
        _, ids = index.search(qv, TOP_K_RETRIEVE)
        t2 = time.perf_counter()
        rerank(q, ids[0].tolist())
        t3 = time.perf_counter()
        embed_t.append(t1 - t0); search_t.append(t2 - t1); ce_t.append(t3 - t2)

embed_t, search_t, ce_t = map(lambda a: np.array(a) * 1000, (embed_t, search_t, ce_t))   # → ms
p1 = embed_t + search_t
p2 = p1 + ce_t


def stats(a):
    return {"mean_ms": a.mean(), "median_ms": np.median(a), "p95_ms": np.percentile(a, 95)}


latency_df = pd.DataFrame({
    "Query embedding (bi-encoder)":         stats(embed_t),
    "FAISS top-10 search":                  stats(search_t),
    "Cross-encoder: score 10 pairs + sort": stats(ce_t),
    "PIPELINE 1 retrieval total":           stats(p1),
    "PIPELINE 2 retrieval total":           stats(p2),
}).T.round(3)
print(f"{len(p1)} timed runs per stage on {DEVICE.upper()}")
display(latency_df)
print(f"Re-ranking overhead: +{p2.mean() - p1.mean():.1f} ms per query "
      f"({p2.mean() / p1.mean():.1f}x the baseline retrieval time)")

1510 timed runs per stage on CPU


,mean_ms,median_ms,p95_ms
Query embedding (bi-encoder),4.722,4.691,5.199
FAISS top-10 search,0.034,0.033,0.036
Cross-encoder: score 10 pairs + sort,44.862,44.600,53.047
PIPELINE 1 retrieval total,4.756,4.725,5.232
PIPELINE 2 retrieval total,49.618,49.343,57.763


Re-ranking overhead: +44.9 ms per query (10.4x the baseline retrieval time)


In [15]:
# LLM generation time per model (from the stress-test runs), and end-to-end time per answer
llm_rows = []
for name in LLMS:
    secs = [a for r in failure_results for a in (r["answers"][name]["s1"], r["answers"][name]["s2"])]
    toks = [a for r in failure_results for a in (r["answers"][name]["tok1"], r["answers"][name]["tok2"])]
    llm_rows.append({"model": name, "answers timed": len(secs), "mean s / answer": np.mean(secs),
                     "p95 s / answer": np.percentile(secs, 95), "tokens / s": sum(toks) / sum(secs)})
llm_df = pd.DataFrame(llm_rows).round(2)
display(llm_df)

e2e = pd.DataFrame([{"model": r["model"],
                     "Pipeline 1 end-to-end (s)": p1.mean() / 1000 + r["mean s / answer"],
                     "Pipeline 2 end-to-end (s)": p2.mean() / 1000 + r["mean s / answer"]} for r in llm_rows])
e2e["re-ranking share of P2 time"] = ((p2.mean() - p1.mean()) / 1000 / e2e["Pipeline 2 end-to-end (s)"]).map("{:.1%}".format)
e2e.round(3)

,model,answers timed,mean s / answer,p95 s / answer,tokens / s
0,Qwen2.5-1.5B,24,3.89,6.03,13.44
1,Qwen3.8-27B,24,1.38,2.18,31.04


,model,Pipeline 1 end-to-end (s),Pipeline 2 end-to-end (s),re-ranking share of P2 time
0,Qwen2.5-1.5B,3.894,3.939,1.1%
1,Qwen3.8-27B,1.382,1.427,3.1%


## 9. Scaling experiment: why not cross-encode the whole collection?

A bi-encoder pays the document-encoding cost **once, offline**. A query then costs one embedding plus a vector
search, which ANN indexes such as HNSW or IVF make sub-linear in N. A cross-encoder has nothing to pre-compute,
because every score depends on the (query, chunk) pair. It must run a full transformer forward pass over **every
chunk for every query**, so cost grows linearly with N. Below I measure both on this document and extrapolate.

In [16]:
q = "Does the six-year maximum duration include semester leaves?"
reps = 5

t0 = time.perf_counter()
for _ in range(reps):
    full_scores = cross_encoder.predict([(q, p) for p in passages], batch_size=32)
ce_full_ms = (time.perf_counter() - t0) / reps * 1000

t0 = time.perf_counter()
for _ in range(reps):
    baseline_retrieve(q)
bi_ms = (time.perf_counter() - t0) / reps * 1000

per_pair_ms = ce_full_ms / len(passages)
print(f"Cross-encoder over ALL {len(passages)} chunks: {ce_full_ms:.0f} ms  ({per_pair_ms:.2f} ms / pair)")
print(f"Bi-encoder retrieval over all chunks:         {bi_ms:.1f} ms")

scale = pd.DataFrame({"corpus_chunks": [len(passages), 10_000, 1_000_000, 100_000_000]})
scale["cross_encoder_all (s/query)"] = (scale.corpus_chunks * per_pair_ms / 1000).round(2)
scale["two-stage: bi top-10 + CE (s/query)"] = round((bi_ms + 10 * per_pair_ms) / 1000, 3)   # ANN search ~constant
scale["CE-all human time"] = pd.to_timedelta(scale["cross_encoder_all (s/query)"], unit="s").astype(str)
print("\nExtrapolation (linear in N for CE; ANN search assumed ~constant for the bi-encoder):")
scale

Cross-encoder over ALL 98 chunks: 511 ms  (5.22 ms / pair)
Bi-encoder retrieval over all chunks:         3.7 ms

Extrapolation (linear in N for CE; ANN search assumed ~constant for the bi-encoder):


,corpus_chunks,cross_encoder_all (s/query),two-stage: bi top-10 + CE (s/query),CE-all human time
0,98,0.51,0.056,0 days 00:00:00.510000
1,10000,52.18,0.056,0 days 00:00:52.180000
2,1000000,5217.65,0.056,0 days 01:26:57.650000
3,100000000,521765.23,0.056,6 days 00:56:05.230000


In [17]:
# Re-rank depth trade-off: how many bi-encoder candidates should the cross-encoder see?
# (depth = all chunks is "cross-encode the whole corpus")
depth_rows = []
for depth in [3, 5, 10, 20, 50, len(passages)]:
    hits, times = [], []
    for _, qq, gold in EVAL_SET:
        t0 = time.perf_counter()
        cand, _ = baseline_retrieve(qq, k=depth)
        rr, _ = rerank(qq, cand)
        times.append(time.perf_counter() - t0)
        hits.append((gold_rank(rr, gold) or 99) <= 3)
    depth_rows.append({"re-rank depth (candidates)": depth, "Hit@3": round(np.mean(hits), 3),
                       "mean retrieval+rerank ms": round(np.mean(times) * 1000, 1)})
depth_df = pd.DataFrame(depth_rows)
depth_df

,re-rank depth (candidates),Hit@3,mean retrieval+rerank ms
0,3,0.854,18.9
1,5,0.881,27.2
2,10,0.907,49.6
3,20,0.907,102.1
4,50,0.914,233.3
5,98,0.914,416.0


## 10. Save results (used for the report)

In [18]:
shift_df.to_csv("results/failure_rank_shifts.csv", index=False)
eval_df.to_csv("results/eval_per_query.csv", index=False)
metrics_df.to_csv("results/eval_metrics.csv")
by_style.to_csv("results/eval_metrics_by_style.csv")
latency_df.to_csv("results/latency_retrieval.csv")
llm_df.to_csv("results/latency_llm.csv", index=False)
e2e.to_csv("results/latency_end_to_end.csv", index=False)
scale.to_csv("results/scaling_extrapolation.csv", index=False)
depth_df.to_csv("results/rerank_depth.csv", index=False)
answer_quality.to_csv("results/answer_correctness.csv")
closed_df.to_csv("results/closed_book.csv", index=False)

with open("results/failure_cases.json", "w", encoding="utf-8") as f:
    json.dump([{k: (v.to_dict(orient="records") if isinstance(v, pd.DataFrame) else v)
                for k, v in r.items()} for r in failure_results], f, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir("results")))

['answer_correctness.csv', 'closed_book.csv', 'eval_metrics.csv', 'eval_metrics_by_style.csv', 'eval_per_query.csv', 'failure_cases.json', 'failure_rank_shifts.csv', 'latency_end_to_end.csv', 'latency_llm.csv', 'latency_retrieval.csv', 'rerank_depth.csv', 'scaling_extrapolation.csv']
